In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

BASIC_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "tabular"
    / "Conjunto_datos_BASICO_AgroCebada2026.csv"
)

basic = pd.read_csv(BASIC_PATH)

print("Shape:", basic.shape)

print("\n=== COLUMNAS ===")
for i, col in enumerate(basic.columns, start=1):
    print(f"{i:03d}. {col}")

print("\n=== PRIMERAS 5 FILAS ===")
display(basic.head())

print("\n=== TIPOS DE DATOS ===")
print(basic.dtypes)

Shape: (107666, 96)

=== COLUMNAS ===
001. ID_POLIGONO
002. fecha_captura
003. sensor
004. ndvi_promedio
005. ndvi_std
006. ndvi_max
007. ndvi_min
008. evi_promedio
009. evi_std
010. evi_max
011. evi_min
012. crc_promedio
013. crc_std
014. crc_max
015. crc_min
016. mcrc_promedio
017. mcrc_std
018. mcrc_max
019. mcrc_min
020. ndsvi_promedio
021. ndsvi_std
022. ndsvi_max
023. ndsvi_min
024. srndi_promedio
025. srndi_std
026. srndi_max
027. srndi_min
028. mra_promedio
029. mra_std
030. mra_max
031. mra_min
032. lai_promedio
033. lai_std
034. lai_max
035. lai_min
036. fapar_promedio
037. fapar_std
038. fapar_max
039. fapar_min
040. savi_promedio
041. savi_std
042. savi_max
043. savi_min
044. vi6t_promedio
045. vi6t_std
046. vi6t_max
047. vi6t_min
048. ndti_promedio
049. ndti_std
050. ndti_max
051. ndti_min
052. sti_promedio
053. sti_std
054. sti_max
055. sti_min
056. ndwi_promedio
057. ndwi_std
058. ndwi_max
059. ndwi_min
060. endwi_promedio
061. endwi_std
062. endwi_max
063. endwi_min
064

,ID_POLIGONO,fecha_captura,sensor,ndvi_promedio,ndvi_std,ndvi_max,ndvi_min,evi_promedio,evi_std,evi_max,...,dswi5_min,nddi_promedio,nddi_std,nddi_max,nddi_min,enddi_promedio,enddi_std,enddi_max,enddi_min,porcentaje_nubosidad
0,AGC_011,20/04/2022,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,AGC_011,16/03/2024,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,AGC_011,15/03/2024,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,AGC_011,26/08/2025,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,AGC_011,10/03/2025,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0



=== TIPOS DE DATOS ===
ID_POLIGONO                 str
fecha_captura               str
sensor                      str
ndvi_promedio           float64
ndvi_std                float64
                         ...   
enddi_promedio          float64
enddi_std               float64
enddi_max               float64
enddi_min               float64
porcentaje_nubosidad    float64
Length: 96, dtype: object


In [4]:
# ============================================
# AUDITORÍA INICIAL DE DATOS FALTANTES
# ============================================

print("=== DATOS FALTANTES POR COLUMNA ===")

missing = pd.DataFrame({
    "missing_count": basic.isna().sum(),
    "missing_pct": basic.isna().mean() * 100
})

missing = missing.sort_values("missing_pct", ascending=False)

display(missing)

=== DATOS FALTANTES POR COLUMNA ===


,missing_count,missing_pct
vi6t_promedio,76618,71.162670
vi6t_min,76618,71.162670
vi6t_max,76618,71.162670
vi6t_std,76618,71.162670
fapar_std,62793,58.322033
...,...,...
enddi_promedio,62789,58.318318
porcentaje_nubosidad,3056,2.838408
sensor,0,0.000000
ID_POLIGONO,0,0.000000


In [5]:
# ============================================
# COMPLETITUD GENERAL DE LAS FILAS
# ============================================

feature_cols = [
    col for col in basic.columns
    if col not in ["ID_POLIGONO", "fecha_captura", "sensor"]
]

basic["n_features_validas"] = basic[feature_cols].notna().sum(axis=1)

print("Total de filas:", len(basic))

print("\n=== FEATURES VÁLIDAS POR FILA ===")
display(
    basic["n_features_validas"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\n=== FILAS COMPLETAMENTE SIN FEATURES ===")
print(
    (basic["n_features_validas"] == 0).sum()
)

print("\n=== FILAS CON AL MENOS UNA FEATURE VÁLIDA ===")
print(
    (basic["n_features_validas"] > 0).sum()
)

Total de filas: 107666

=== FEATURES VÁLIDAS POR FILA ===


count    107666.000000
mean         38.804841
std          42.464774
min           0.000000
1%            0.000000
5%            1.000000
10%           1.000000
25%           1.000000
50%           5.000000
75%          89.000000
90%          89.000000
95%          89.000000
99%          89.000000
max          89.000000
Name: n_features_validas, dtype: float64


=== FILAS COMPLETAMENTE SIN FEATURES ===
3056

=== FILAS CON AL MENOS UNA FEATURE VÁLIDA ===
104610


In [6]:
# ============================================
# COMPLETITUD POR SENSOR
# ============================================

sensor_completeness = (
    basic.groupby("sensor")["n_features_validas"]
    .agg(
        filas="count",
        media_features_validas="mean",
        mediana_features_validas="median",
        min_features_validas="min",
        max_features_validas="max"
    )
)

display(sensor_completeness)

,filas,media_features_validas,mediana_features_validas,min_features_validas,max_features_validas
sensor,,,,,
Landsat,48626,3.491178,5.0,0,5
Sentinel-2,59040,67.889566,89.0,1,89


In [7]:
# ============================================
# AUDITORÍA DE FILAS SIN FEATURES
# ============================================

empty_rows = basic[basic["n_features_validas"] == 0].copy()

print("Filas completamente sin features:", len(empty_rows))

print("\n=== POR SENSOR ===")
display(
    empty_rows["sensor"]
    .value_counts(dropna=False)
)

print("\n=== NÚMERO DE PARCELAS ===")
print(
    empty_rows["ID_POLIGONO"].nunique()
)

print("\n=== FECHAS ===")
print(
    empty_rows["fecha_captura"].nunique()
)

print("\n=== RANGO DE FECHAS ===")
print(
    empty_rows["fecha_captura"].min(),
    "→",
    empty_rows["fecha_captura"].max()
)

print("\n=== PRIMERAS FILAS ===")
display(
    empty_rows[
        ["ID_POLIGONO", "fecha_captura", "sensor", "porcentaje_nubosidad"]
    ].head(20)
)

Filas completamente sin features: 3056

=== POR SENSOR ===


sensor
Landsat    3056
Name: count, dtype: int64


=== NÚMERO DE PARCELAS ===
40

=== FECHAS ===
195

=== RANGO DE FECHAS ===
01/01/2023 → 31/08/2024

=== PRIMERAS FILAS ===


,ID_POLIGONO,fecha_captura,sensor,porcentaje_nubosidad
1274,AGC_061,01/05/2023,Landsat,NaN
3511,AGC_115,12/04/2022,Landsat,NaN
3512,AGC_115,14/03/2023,Landsat,NaN
3514,AGC_115,13/02/2024,Landsat,NaN
3515,AGC_115,13/10/2022,Landsat,NaN
3516,AGC_115,14/01/2025,Landsat,NaN
3517,AGC_115,01/07/2025,Landsat,NaN
3518,AGC_115,21/10/2022,Landsat,NaN
3519,AGC_115,06/09/2023,Landsat,NaN
3520,AGC_115,30/07/2024,Landsat,NaN


In [8]:
# ============================================
# ¿QUÉ VARIABLES TIENE REALMENTE LANDSAT?
# ============================================

land = basic[basic["sensor"] == "Landsat"].copy()

feature_cols = [
    col for col in basic.columns
    if col not in ["ID_POLIGONO", "fecha_captura", "sensor"]
]

land_valid = pd.DataFrame({
    "valid_count": land[feature_cols].notna().sum(),
    "valid_pct": land[feature_cols].notna().mean() * 100
})

land_valid = land_valid.sort_values("valid_pct", ascending=False)

display(land_valid)

,valid_count,valid_pct
n_features_validas,48626,100.000000
porcentaje_nubosidad,45570,93.715296
vi6t_min,31048,63.850615
vi6t_max,31048,63.850615
vi6t_std,31048,63.850615
...,...,...
nddi_min,0,0.000000
enddi_std,0,0.000000
enddi_promedio,0,0.000000
enddi_min,0,0.000000


In [9]:
# ============================================
# AUDITORÍA DE VI6T - LANDSAT
# ============================================

vi6t_cols = [
    "vi6t_promedio",
    "vi6t_std",
    "vi6t_min",
    "vi6t_max"
]

land_vi6t = land[vi6t_cols]

print("=== ESTADÍSTICAS VI6T LANDSAT ===")
display(
    land_vi6t.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\n=== VALORES NO FINITOS ===")
print(
    "Infinitos:",
    np.isinf(land_vi6t.select_dtypes(include=np.number)).sum().sum()
)

print("\n=== RELACIONES ENTRE ESTADÍSTICOS ===")

print(
    "Filas donde min > promedio:",
    (land["vi6t_min"] > land["vi6t_promedio"]).sum()
)

print(
    "Filas donde promedio > max:",
    (land["vi6t_promedio"] > land["vi6t_max"]).sum()
)

print(
    "Filas donde std < 0:",
    (land["vi6t_std"] < 0).sum()
)

=== ESTADÍSTICAS VI6T LANDSAT ===


,vi6t_promedio,vi6t_std,vi6t_min,vi6t_max
count,31048.000000,31048.000000,31048.000000,31048.000000
mean,-0.620601,0.011757,-0.646256,-0.591942
std,0.025795,0.006147,0.025215,0.033510
min,-0.802000,0.000000,-0.880000,-0.802000
1%,-0.664000,0.003000,-0.691000,-0.647000
5%,-0.654000,0.004000,-0.680000,-0.635000
25%,-0.638000,0.007000,-0.663000,-0.614000
50%,-0.625000,0.010000,-0.649000,-0.596000
75%,-0.608000,0.015000,-0.634000,-0.576000
95%,-0.574000,0.023000,-0.600350,-0.534000



=== VALORES NO FINITOS ===
Infinitos: 0

=== RELACIONES ENTRE ESTADÍSTICOS ===
Filas donde min > promedio: 0
Filas donde promedio > max: 0
Filas donde std < 0: 0


In [10]:
# ============================================
# AUDITORÍA DE VARIABLES - SENTINEL-2
# ============================================

sentinel = basic[basic["sensor"] == "Sentinel-2"].copy()

feature_cols = [
    col for col in basic.columns
    if col not in ["ID_POLIGONO", "fecha_captura", "sensor"]
]

sentinel_valid = pd.DataFrame({
    "valid_count": sentinel[feature_cols].notna().sum(),
    "valid_pct": sentinel[feature_cols].notna().mean() * 100
})

sentinel_valid = sentinel_valid.sort_values(
    "valid_pct",
    ascending=False
)

display(sentinel_valid)

,valid_count,valid_pct
porcentaje_nubosidad,59040,100.000000
n_features_validas,59040,100.000000
ndvi_max,44877,76.011179
ndvi_min,44877,76.011179
ndvi_promedio,44877,76.011179
...,...,...
fapar_std,44873,76.004404
vi6t_std,0,0.000000
vi6t_promedio,0,0.000000
vi6t_min,0,0.000000


In [11]:
print("Filas Sentinel-2:", len(sentinel))

print("\nFilas con todas las features disponibles:")
print(
    (sentinel[feature_cols].notna().sum(axis=1) == len(feature_cols)).sum()
)

print("\nFilas con al menos una feature disponible:")
print(
    (sentinel[feature_cols].notna().sum(axis=1) > 0).sum()
)

print("\nNúmero de parcelas:")
print(sentinel["ID_POLIGONO"].nunique())

print("\nNúmero de fechas:")
print(sentinel["fecha_captura"].nunique())

Filas Sentinel-2: 59040

Filas con todas las features disponibles:
0

Filas con al menos una feature disponible:
59040

Número de parcelas:
197

Número de fechas:
301


In [12]:
# ============================================
# FRECUENCIA TEMPORAL SENTINEL-2
# ============================================

sentinel = basic[basic["sensor"] == "Sentinel-2"].copy()

sentinel["fecha_captura_dt"] = pd.to_datetime(
    sentinel["fecha_captura"],
    dayfirst=True,
    errors="coerce"
)

print("=== RANGO TEMPORAL ===")
print(
    sentinel["fecha_captura_dt"].min(),
    "→",
    sentinel["fecha_captura_dt"].max()
)

print("\n=== AÑOS ===")
display(
    sentinel["fecha_captura_dt"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\n=== OBSERVACIONES POR MES ===")
display(
    sentinel["fecha_captura_dt"]
    .dt.month
    .value_counts()
    .sort_index()
)

print("\n=== FECHAS ÚNICAS POR AÑO ===")
display(
    sentinel.groupby(
        sentinel["fecha_captura_dt"].dt.year
    )["fecha_captura_dt"]
    .nunique()
)

print("\n=== FECHAS ÚNICAS TOTALES ===")
print(
    sentinel["fecha_captura_dt"].nunique()
)

=== RANGO TEMPORAL ===
2022-01-01 00:00:00 → 2025-12-31 00:00:00

=== AÑOS ===


fecha_captura_dt
2022    13922
2023    13917
2024    14381
2025    16820
Name: count, dtype: int64


=== OBSERVACIONES POR MES ===


fecha_captura_dt
1     5319
2     3940
3     5319
4     5057
5     5087
6     5122
7     4925
8     4925
9     4632
10    4728
11    4864
12    5122
Name: count, dtype: int64


=== FECHAS ÚNICAS POR AÑO ===


fecha_captura_dt
2022    71
2023    71
2024    73
2025    86
Name: fecha_captura_dt, dtype: int64


=== FECHAS ÚNICAS TOTALES ===
301


In [13]:
# ============================================
# COBERTURA TEMPORAL POR PARCELA
# ============================================

sentinel["anio"] = sentinel["fecha_captura_dt"].dt.year

parcel_year = (
    sentinel
    .groupby(["ID_POLIGONO", "anio"])
    .agg(
        observaciones=("fecha_captura_dt", "size"),
        fechas_unicas=("fecha_captura_dt", "nunique")
    )
    .reset_index()
)

print("=== OBSERVACIONES POR PARCELA Y AÑO ===")
display(
    parcel_year.groupby("anio")["observaciones"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\n=== FECHAS ÚNICAS POR PARCELA Y AÑO ===")
display(
    parcel_year.groupby("anio")["fechas_unicas"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

=== OBSERVACIONES POR PARCELA Y AÑO ===


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
anio,,,,,,,,,,,,
2022,197.0,70.670051,0.471392,70.0,70.0,70.0,70.0,71.0,71.0,71.0,71.0,71.0
2023,197.0,70.644670,0.766409,69.0,69.0,69.0,71.0,71.0,71.0,71.0,71.0,71.0
2024,197.0,73.000000,0.000000,73.0,73.0,73.0,73.0,73.0,73.0,73.0,73.0,73.0
2025,197.0,85.380711,0.927050,84.0,84.0,84.0,84.0,86.0,86.0,86.0,86.0,86.0



=== FECHAS ÚNICAS POR PARCELA Y AÑO ===


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
anio,,,,,,,,,,,,
2022,197.0,70.670051,0.471392,70.0,70.0,70.0,70.0,71.0,71.0,71.0,71.0,71.0
2023,197.0,70.644670,0.766409,69.0,69.0,69.0,71.0,71.0,71.0,71.0,71.0,71.0
2024,197.0,73.000000,0.000000,73.0,73.0,73.0,73.0,73.0,73.0,73.0,73.0,73.0
2025,197.0,85.380711,0.927050,84.0,84.0,84.0,84.0,86.0,86.0,86.0,86.0,86.0


In [14]:
print("=== PARCELAS CON DATOS EN CADA AÑO ===")

for year in sorted(sentinel["anio"].unique()):
    n = sentinel.loc[
        sentinel["anio"] == year,
        "ID_POLIGONO"
    ].nunique()

    print(f"{year}: {n} parcelas")

=== PARCELAS CON DATOS EN CADA AÑO ===
2022: 197 parcelas
2023: 197 parcelas
2024: 197 parcelas
2025: 197 parcelas


In [15]:
# ============================================
# NUBOSIDAD VS DISPONIBILIDAD DE NDVI
# ============================================

sentinel_ndvi = sentinel[
    [
        "ID_POLIGONO",
        "fecha_captura_dt",
        "porcentaje_nubosidad",
        "ndvi_promedio"
    ]
].copy()

sentinel_ndvi["ndvi_disponible"] = (
    sentinel_ndvi["ndvi_promedio"].notna()
)

print("=== NDVI DISPONIBLE ===")
display(
    sentinel_ndvi["ndvi_disponible"]
    .value_counts()
)

print("\n=== NUBOSIDAD: OBSERVACIONES CON NDVI ===")
display(
    sentinel_ndvi.loc[
        sentinel_ndvi["ndvi_disponible"],
        "porcentaje_nubosidad"
    ].describe()
)

print("\n=== NUBOSIDAD: OBSERVACIONES SIN NDVI ===")
display(
    sentinel_ndvi.loc[
        ~sentinel_ndvi["ndvi_disponible"],
        "porcentaje_nubosidad"
    ].describe()
)

=== NDVI DISPONIBLE ===


ndvi_disponible
True     44877
False    14163
Name: count, dtype: int64


=== NUBOSIDAD: OBSERVACIONES CON NDVI ===


count    44877.000000
mean         6.846218
std         20.350767
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         96.000000
Name: porcentaje_nubosidad, dtype: float64


=== NUBOSIDAD: OBSERVACIONES SIN NDVI ===


count    14163.0
mean       100.0
std          0.0
min        100.0
25%        100.0
50%        100.0
75%        100.0
max        100.0
Name: porcentaje_nubosidad, dtype: float64

In [16]:
# ============================================
# RELACIÓN NUBOSIDAD - DISPONIBILIDAD
# PARA TODOS LOS ÍNDICES SENTINEL-2
# ============================================

index_cols = [
    col for col in sentinel.columns
    if col not in [
        "ID_POLIGONO",
        "fecha_captura",
        "fecha_captura_dt",
        "sensor",
        "porcentaje_nubosidad"
    ]
]

rows = []

for col in index_cols:
    available = sentinel[col].notna()

    rows.append({
        "variable": col,
        "total": len(sentinel),
        "disponibles": available.sum(),
        "faltantes": (~available).sum(),
        "cobertura_pct": available.mean() * 100,
        "nubosidad_media_disponible": sentinel.loc[
            available,
            "porcentaje_nubosidad"
        ].mean(),
        "nubosidad_media_faltante": sentinel.loc[
            ~available,
            "porcentaje_nubosidad"
        ].mean()
    })

availability_audit = (
    pd.DataFrame(rows)
    .sort_values("cobertura_pct")
)

display(availability_audit)

,variable,total,disponibles,faltantes,cobertura_pct,nubosidad_media_disponible,nubosidad_media_faltante
40,vi6t_promedio,59040,0,59040,0.000000,NaN,29.192712
41,vi6t_std,59040,0,59040,0.000000,NaN,29.192712
42,vi6t_max,59040,0,59040,0.000000,NaN,29.192712
43,vi6t_min,59040,0,59040,0.000000,NaN,29.192712
35,fapar_min,59040,44873,14167,76.004404,6.846828,99.971765
...,...,...,...,...,...,...,...
89,enddi_std,59040,44877,14163,76.011179,6.846218,100.000000
90,enddi_max,59040,44877,14163,76.011179,6.846218,100.000000
91,enddi_min,59040,44877,14163,76.011179,6.846218,100.000000
92,n_features_validas,59040,59040,0,100.000000,29.192712,NaN


In [17]:
# Eliminar únicamente columnas auxiliares creadas durante la auditoría
cols_auxiliares = [
    "n_features_validas",
    "fecha_captura_dt",
    "anio"
]

basic = basic.drop(
    columns=[c for c in cols_auxiliares if c in basic.columns]
)

print("Shape:", basic.shape)
print("Columnas:", len(basic.columns))

Shape: (107666, 96)
Columnas: 96


In [18]:
# ============================================
# AUDITORÍA DE RANGOS - ÍNDICES PRINCIPALES
# ============================================

indices_principales = [
    "ndvi",
    "evi",
    "lai",
    "fapar",
    "ndwi",
    "msi"
]

columnas_auditar = []

for indice in indices_principales:
    for estadistico in ["promedio", "std", "min", "max"]:
        columnas_auditar.append(f"{indice}_{estadistico}")

sentinel_valid = sentinel[columnas_auditar].copy()

display(
    sentinel_valid.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
ndvi_promedio,44877.0,0.242993,0.157198,-0.066,0.05400,0.104,0.138,0.175,0.299,0.603,0.70924,0.820
ndvi_std,44877.0,0.046832,0.030247,0.002,0.00700,0.011,0.023,0.041,0.063,0.104,0.14000,0.258
ndvi_min,44877.0,0.140941,0.098607,-0.110,-0.03500,0.002,0.091,0.115,0.171,0.350,0.47000,0.685
ndvi_max,44877.0,0.414484,0.192034,0.002,0.13300,0.168,0.252,0.379,0.556,0.773,0.83300,0.890
evi_promedio,44877.0,0.190303,0.142251,-0.031,0.03800,0.078,0.103,0.130,0.219,0.520,0.68124,0.905
evi_std,44877.0,0.042548,0.034267,0.002,0.00600,0.009,0.018,0.032,0.056,0.113,0.16600,0.317
evi_min,44877.0,0.103929,0.077956,-0.921,-0.02100,0.001,0.064,0.084,0.123,0.267,0.37900,0.758
evi_max,44877.0,0.353817,0.208074,0.000,0.11100,0.134,0.199,0.286,0.451,0.804,0.96500,1.000
lai_promedio,44877.0,0.131016,0.284691,-0.192,-0.13000,-0.069,-0.027,0.016,0.161,0.752,1.24524,3.035
lai_std,44877.0,0.080122,0.098064,0.003,0.00900,0.013,0.027,0.047,0.090,0.264,0.49524,1.330


In [19]:
# ============================================
# CONSISTENCIA INTERNA DE LOS ESTADÍSTICOS
# ============================================

indices_principales = [
    "ndvi",
    "evi",
    "lai",
    "fapar",
    "ndwi",
    "msi"
]

resultados = []

for indice in indices_principales:

    mean_col = f"{indice}_promedio"
    std_col = f"{indice}_std"
    min_col = f"{indice}_min"
    max_col = f"{indice}_max"

    subset = sentinel[
        [mean_col, std_col, min_col, max_col]
    ].dropna()

    resultados.append({
        "indice": indice,
        "filas_evaluadas": len(subset),
        "min_mayor_promedio": (
            subset[min_col] > subset[mean_col]
        ).sum(),
        "promedio_mayor_max": (
            subset[mean_col] > subset[max_col]
        ).sum(),
        "std_negativa": (
            subset[std_col] < 0
        ).sum()
    })

consistencia = pd.DataFrame(resultados)

display(consistencia)

,indice,filas_evaluadas,min_mayor_promedio,promedio_mayor_max,std_negativa
0,ndvi,44877,0,0,0
1,evi,44877,0,0,0
2,lai,44877,0,0,0
3,fapar,44873,0,0,0
4,ndwi,44877,0,0,0
5,msi,44877,0,0,0


In [20]:
# ============================================
# DUPLICADOS DE OBSERVACIONES SATELITALES
# ============================================

keys = [
    "ID_POLIGONO",
    "fecha_captura",
    "sensor"
]

duplicados = (
    basic
    .groupby(keys)
    .size()
    .reset_index(name="n_registros")
)

duplicados = duplicados[
    duplicados["n_registros"] > 1
].sort_values(
    "n_registros",
    ascending=False
)

print("=== COMBINACIONES DUPLICADAS ===")
print("Número de combinaciones duplicadas:", len(duplicados))

print("\n=== REGISTROS INVOLUCRADOS ===")
print(
    duplicados["n_registros"].sum()
    if len(duplicados) > 0
    else 0
)

display(duplicados.head(20))

=== COMBINACIONES DUPLICADAS ===
Número de combinaciones duplicadas: 0

=== REGISTROS INVOLUCRADOS ===
0


,ID_POLIGONO,fecha_captura,sensor,n_registros


In [21]:
# ============================================
# COBERTURA SENTINEL-2 POR FECHA
# ============================================

sentinel_2025 = sentinel[
    sentinel["fecha_captura_dt"].dt.year == 2025
].copy()

cobertura_fecha = (
    sentinel_2025
    .groupby("fecha_captura_dt")
    .agg(
        parcelas=("ID_POLIGONO", "nunique"),
        ndvi_validos=("ndvi_promedio", "count"),
        nubosidad_media=("porcentaje_nubosidad", "mean")
    )
    .reset_index()
)

cobertura_fecha["pct_parcelas_ndvi"] = (
    cobertura_fecha["ndvi_validos"] / 197 * 100
)

display(
    cobertura_fecha
    .sort_values("fecha_captura_dt")
)

,fecha_captura_dt,parcelas,ndvi_validos,nubosidad_media,pct_parcelas_ndvi
0,2025-01-05,197,0,100.000000,0.000000
1,2025-01-10,197,139,47.988528,70.558376
2,2025-01-15,197,197,0.017197,100.000000
3,2025-01-20,197,195,10.087784,98.984772
4,2025-01-25,197,196,1.288960,99.492386
...,...,...,...,...,...
81,2025-12-13,197,58,80.392598,29.441624
82,2025-12-16,197,162,21.814114,82.233503
83,2025-12-21,197,139,37.868135,70.558376
84,2025-12-26,197,174,16.040810,88.324873


In [22]:
# ============================================
# COBERTURA SENTINEL-2 POR FECHA
# ============================================

sentinel_2025 = sentinel[
    sentinel["fecha_captura_dt"].dt.year == 2025
].copy()

cobertura_fecha = (
    sentinel_2025
    .groupby("fecha_captura_dt")
    .agg(
        parcelas=("ID_POLIGONO", "nunique"),
        ndvi_validos=("ndvi_promedio", "count"),
        nubosidad_media=("porcentaje_nubosidad", "mean")
    )
    .reset_index()
)

cobertura_fecha["pct_parcelas_ndvi"] = (
    cobertura_fecha["ndvi_validos"] / 197 * 100
)

display(
    cobertura_fecha
    .sort_values("fecha_captura_dt")
)

,fecha_captura_dt,parcelas,ndvi_validos,nubosidad_media,pct_parcelas_ndvi
0,2025-01-05,197,0,100.000000,0.000000
1,2025-01-10,197,139,47.988528,70.558376
2,2025-01-15,197,197,0.017197,100.000000
3,2025-01-20,197,195,10.087784,98.984772
4,2025-01-25,197,196,1.288960,99.492386
...,...,...,...,...,...
81,2025-12-13,197,58,80.392598,29.441624
82,2025-12-16,197,162,21.814114,82.233503
83,2025-12-21,197,139,37.868135,70.558376
84,2025-12-26,197,174,16.040810,88.324873


In [23]:
# ============================================
# COBERTURA TEMPORAL ÚTIL - SENTINEL-2 2025
# ============================================

print("=== FECHAS SEGÚN COBERTURA DE NDVI ===")

print(
    "Cobertura >= 90%:",
    (cobertura_fecha["pct_parcelas_ndvi"] >= 90).sum()
)

print(
    "Cobertura >= 75%:",
    (cobertura_fecha["pct_parcelas_ndvi"] >= 75).sum()
)

print(
    "Cobertura >= 50%:",
    (cobertura_fecha["pct_parcelas_ndvi"] >= 50).sum()
)

print(
    "Cobertura < 50%:",
    (cobertura_fecha["pct_parcelas_ndvi"] < 50).sum()
)

print("\n=== ESTADÍSTICAS DE COBERTURA ===")

display(
    cobertura_fecha["pct_parcelas_ndvi"]
    .describe(
        percentiles=[
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

=== FECHAS SEGÚN COBERTURA DE NDVI ===
Cobertura >= 90%: 48
Cobertura >= 75%: 60
Cobertura >= 50%: 70
Cobertura < 50%: 16

=== ESTADÍSTICAS DE COBERTURA ===


count     86.000000
mean      77.269508
std       32.871847
min        0.000000
5%         0.507614
10%       17.005076
25%       69.416244
50%       95.939086
75%      100.000000
90%      100.000000
95%      100.000000
max      100.000000
Name: pct_parcelas_ndvi, dtype: float64

In [24]:
# ============================================
# AUDITORÍA INICIAL - DATASET PRO
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

PRO_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "tabular"
    / "Conjunto_datos_PRO_AgroCebada.csv"
)

pro = pd.read_csv(PRO_PATH)

print("Shape:", pro.shape)

print("\n=== COLUMNAS ===")
for i, col in enumerate(pro.columns, start=1):
    print(f"{i:03d}. {col}")

print("\n=== PRIMERAS 5 FILAS ===")
display(pro.head())

print("\n=== TIPOS DE DATOS ===")
print(pro.dtypes)

Shape: (47804, 20)

=== COLUMNAS ===
001. ID_POLIGONO
002. fecha_captura
003. sensor
004. ndvi_promedio
005. ndvi_std
006. ndvi_max
007. ndvi_min
008. evi_promedio
009. evi_std
010. evi_max
011. evi_min
012. lai_promedio
013. lai_std
014. lai_max
015. lai_min
016. msavi_promedio
017. msavi_std
018. msavi_max
019. msavi_min
020. porcentaje_nubosidad

=== PRIMERAS 5 FILAS ===


,ID_POLIGONO,fecha_captura,sensor,ndvi_promedio,ndvi_std,ndvi_max,ndvi_min,evi_promedio,evi_std,evi_max,evi_min,lai_promedio,lai_std,lai_max,lai_min,msavi_promedio,msavi_std,msavi_max,msavi_min,porcentaje_nubosidad
0,AGC_056,05/02/2025,Planet,0.300,0.042,0.553,0.175,0.138,0.022,0.251,0.068,1.049,0.182,2.144,0.586,0.135,0.022,0.244,0.066,0.0
1,AGC_056,08/02/2025,Planet,0.204,0.043,0.529,0.125,0.110,0.018,0.209,0.074,0.697,0.167,1.888,0.247,0.110,0.018,0.207,0.072,0.0
2,AGC_056,07/02/2025,Planet,0.241,0.043,0.534,0.154,0.134,0.018,0.224,0.098,0.826,0.159,1.858,0.404,0.135,0.018,0.230,0.098,0.0
3,AGC_056,09/02/2025,Planet,0.246,0.039,0.536,0.153,0.142,0.018,0.243,0.098,0.818,0.159,1.849,0.399,0.141,0.018,0.237,0.095,0.0
4,AGC_056,30/01/2025,Planet,0.211,0.046,0.533,0.113,0.110,0.020,0.229,0.069,0.724,0.187,1.955,0.277,0.111,0.021,0.230,0.067,0.0



=== TIPOS DE DATOS ===
ID_POLIGONO                 str
fecha_captura               str
sensor                      str
ndvi_promedio           float64
ndvi_std                float64
ndvi_max                float64
ndvi_min                float64
evi_promedio            float64
evi_std                 float64
evi_max                 float64
evi_min                 float64
lai_promedio            float64
lai_std                 float64
lai_max                 float64
lai_min                 float64
msavi_promedio          float64
msavi_std               float64
msavi_max               float64
msavi_min               float64
porcentaje_nubosidad    float64
dtype: object


In [25]:
# ============================================
# AUDITORÍA TEMPORAL - PRO / PLANET
# ============================================

pro["fecha_captura_dt"] = pd.to_datetime(
    pro["fecha_captura"],
    dayfirst=True,
    errors="coerce"
)

print("=== RANGO TEMPORAL ===")
print(
    pro["fecha_captura_dt"].min(),
    "→",
    pro["fecha_captura_dt"].max()
)

print("\n=== NÚMERO DE PARCELAS ===")
print(
    pro["ID_POLIGONO"].nunique()
)

print("\n=== NÚMERO DE FECHAS ÚNICAS ===")
print(
    pro["fecha_captura_dt"].nunique()
)

print("\n=== OBSERVACIONES POR MES ===")
display(
    pro["fecha_captura_dt"]
    .dt.month
    .value_counts()
    .sort_index()
)

print("\n=== OBSERVACIONES POR MES Y AÑO ===")
display(
    pro.groupby(
        [
            pro["fecha_captura_dt"].dt.year,
            pro["fecha_captura_dt"].dt.month
        ]
    )
    .size()
    .rename("observaciones")
)

=== RANGO TEMPORAL ===
2025-01-01 00:00:00 → 2025-12-30 00:00:00

=== NÚMERO DE PARCELAS ===
197

=== NÚMERO DE FECHAS ÚNICAS ===
311

=== OBSERVACIONES POR MES ===


fecha_captura_dt
1     4316
2     4135
3     5166
4     4760
5     4436
6     2213
7     2940
8     4381
9     3964
10    3561
11    3407
12    4525
Name: count, dtype: int64


=== OBSERVACIONES POR MES Y AÑO ===


fecha_captura_dt  fecha_captura_dt
2025              1                   4316
                  2                   4135
                  3                   5166
                  4                   4760
                  5                   4436
                  6                   2213
                  7                   2940
                  8                   4381
                  9                   3964
                  10                  3561
                  11                  3407
                  12                  4525
Name: observaciones, dtype: int64

In [26]:
print("=== FECHAS ÚNICAS POR MES ===")

fechas_mes = (
    pro
    .groupby(
        [
            pro["fecha_captura_dt"].dt.year,
            pro["fecha_captura_dt"].dt.month
        ]
    )["fecha_captura_dt"]
    .nunique()
    .rename("fechas_unicas")
)

display(fechas_mes)

=== FECHAS ÚNICAS POR MES ===


fecha_captura_dt  fecha_captura_dt
2025              1                   28
                  2                   24
                  3                   31
                  4                   29
                  5                   25
                  6                   18
                  7                   22
                  8                   27
                  9                   29
                  10                  26
                  11                  25
                  12                  27
Name: fechas_unicas, dtype: int64

In [27]:
pro_cobertura_parcela = (
    pro.groupby("ID_POLIGONO")
       .agg(
           observaciones=("fecha_captura_dt", "size"),
           fechas_unicas=("fecha_captura_dt", "nunique"),
           fecha_inicio=("fecha_captura_dt", "min"),
           fecha_fin=("fecha_captura_dt", "max")
       )
       .reset_index()
)

print("=== COBERTURA PRO POR PARCELA ===")
print(pro_cobertura_parcela.describe(include="all"))

print("\n=== PARCELAS CON MENOS OBSERVACIONES ===")
print(
    pro_cobertura_parcela
    .sort_values("observaciones")
    .head(15)
    .to_string(index=False)
)

print("\n=== PARCELAS CON MÁS OBSERVACIONES ===")
print(
    pro_cobertura_parcela
    .sort_values("observaciones", ascending=False)
    .head(15)
    .to_string(index=False)
)

=== COBERTURA PRO POR PARCELA ===
       ID_POLIGONO  observaciones  fechas_unicas                fecha_inicio  \
count          197     197.000000     197.000000                         197   
unique         197            NaN            NaN                         NaN   
top        AGC_001            NaN            NaN                         NaN   
freq             1            NaN            NaN                         NaN   
mean           NaN     242.659898     242.659898  2025-01-01 04:30:27.411167   
min            NaN     233.000000     233.000000         2025-01-01 00:00:00   
25%            NaN     240.000000     240.000000         2025-01-01 00:00:00   
50%            NaN     243.000000     243.000000         2025-01-01 00:00:00   
75%            NaN     246.000000     246.000000         2025-01-01 00:00:00   
max            NaN     251.000000     251.000000         2025-01-02 00:00:00   
std            NaN       4.418314       4.418314                         NaN   

     

In [28]:
indices_pro = [
    "ndvi",
    "evi",
    "lai",
    "msavi"
]

print("=== DISPONIBILIDAD DE ÍNDICES PRO ===\n")

for indice in indices_pro:
    print(f"--- {indice.upper()} ---")
    
    for estadistico in ["promedio", "std", "max", "min"]:
        col = f"{indice}_{estadistico}"
        
        disponibles = pro[col].notna().sum()
        faltantes = pro[col].isna().sum()
        porcentaje = disponibles / len(pro) * 100
        
        print(
            f"{estadistico:10s} | "
            f"disponibles: {disponibles:5d} | "
            f"faltantes: {faltantes:5d} | "
            f"cobertura: {porcentaje:6.2f}%"
        )
    
    print()

=== DISPONIBILIDAD DE ÍNDICES PRO ===

--- NDVI ---
promedio   | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
std        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
max        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
min        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%

--- EVI ---
promedio   | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
std        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
max        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
min        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%

--- LAI ---
promedio   | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
std        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
max        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%
min        | disponibles: 35747 | faltantes: 12057 | cobertura:  74.78%

--- MSAVI ---
promedio   | disponibles: 35747 | faltantes

In [29]:
print("=== DISPONIBILIDAD CONJUNTA DE LOS 4 ÍNDICES ===")

cols_indices = [
    "ndvi_promedio",
    "evi_promedio",
    "lai_promedio",
    "msavi_promedio"
]

todos_disponibles = pro[cols_indices].notna().all(axis=1)

print(f"\nFilas con los 4 índices disponibles: {todos_disponibles.sum()}")
print(f"Filas con al menos un índice faltante: {(~todos_disponibles).sum()}")
print(f"Cobertura conjunta: {todos_disponibles.mean()*100:.2f}%")

=== DISPONIBILIDAD CONJUNTA DE LOS 4 ÍNDICES ===

Filas con los 4 índices disponibles: 35747
Filas con al menos un índice faltante: 12057
Cobertura conjunta: 74.78%


In [30]:
cols_indices = [
    "ndvi_promedio",
    "evi_promedio",
    "lai_promedio",
    "msavi_promedio"
]

todos_disponibles = pro[cols_indices].notna().all(axis=1)

print("=== NUBOSIDAD SEGÚN DISPONIBILIDAD DE ÍNDICES ===\n")

print("CON ÍNDICES DISPONIBLES:")
print(
    pro.loc[todos_disponibles, "porcentaje_nubosidad"]
       .describe()
)

print("\nSIN ÍNDICES:")
print(
    pro.loc[~todos_disponibles, "porcentaje_nubosidad"]
       .describe()
)

=== NUBOSIDAD SEGÚN DISPONIBILIDAD DE ÍNDICES ===

CON ÍNDICES DISPONIBLES:
count    35747.000000
mean         6.527249
std         20.244425
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         95.996000
Name: porcentaje_nubosidad, dtype: float64

SIN ÍNDICES:
count    12057.0
mean       100.0
std          0.0
min        100.0
25%        100.0
50%        100.0
75%        100.0
max        100.0
Name: porcentaje_nubosidad, dtype: float64


In [31]:
print("\n=== DISTRIBUCIÓN DE NUBOSIDAD ===")

bins = [-0.01, 0, 10, 25, 50, 75, 90, 100.01]
labels = [
    "0%",
    ">0–10%",
    ">10–25%",
    ">25–50%",
    ">50–75%",
    ">75–90%",
    ">90–100%"
]

pro_temp = pro.copy()

pro_temp["grupo_nubosidad"] = pd.cut(
    pro_temp["porcentaje_nubosidad"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

pro_temp["indices_completos"] = todos_disponibles

tabla_nubosidad = pd.crosstab(
    pro_temp["grupo_nubosidad"],
    pro_temp["indices_completos"],
    normalize="index"
) * 100

print(tabla_nubosidad)


=== DISTRIBUCIÓN DE NUBOSIDAD ===
indices_completos      False       True 
grupo_nubosidad                         
0%                  0.000000  100.000000
>0–10%              0.000000  100.000000
>10–25%             0.000000  100.000000
>25–50%             0.000000  100.000000
>50–75%             0.000000  100.000000
>75–90%             0.000000  100.000000
>90–100%           94.683524    5.316476


In [32]:
# ============================================================
# AUDITORÍA: NUBOSIDAD VS ÍNDICES PRO
# ============================================================

indices_pro = ["ndvi", "evi", "lai", "msavi"]

# Solo observaciones donde los 4 índices están disponibles
cols_indices = [
    "ndvi_promedio",
    "evi_promedio",
    "lai_promedio",
    "msavi_promedio"
]

indices_completos = pro[cols_indices].notna().all(axis=1)

pro_validos = pro.loc[indices_completos].copy()

# Categorías de nubosidad
bins = [-0.01, 0, 10, 25, 50, 75, 90, 100]
labels = [
    "0%",
    ">0–10%",
    ">10–25%",
    ">25–50%",
    ">50–75%",
    ">75–90%",
    ">90–100%"
]

pro_validos["grupo_nubosidad"] = pd.cut(
    pro_validos["porcentaje_nubosidad"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("=== NÚMERO DE OBSERVACIONES POR GRUPO DE NUBOSIDAD ===\n")

print(
    pro_validos["grupo_nubosidad"]
    .value_counts(sort=False)
)

print("\n\n=== ESTADÍSTICAS DE LOS ÍNDICES POR NUBOSIDAD ===\n")

tabla_nubosidad_indices = (
    pro_validos
    .groupby("grupo_nubosidad", observed=False)[
        [
            "ndvi_promedio",
            "evi_promedio",
            "lai_promedio",
            "msavi_promedio"
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
)

print(tabla_nubosidad_indices)

=== NÚMERO DE OBSERVACIONES POR GRUPO DE NUBOSIDAD ===

grupo_nubosidad
0%          30317
>0–10%       1391
>10–25%       763
>25–50%       967
>50–75%       897
>75–90%       735
>90–100%      677
Name: count, dtype: int64


=== ESTADÍSTICAS DE LOS ÍNDICES POR NUBOSIDAD ===

                ndvi_promedio                                   evi_promedio  \
                        count      mean       std    min    max        count   
grupo_nubosidad                                                                
0%                      30317  0.342567  0.168112  0.072  0.907        30317   
>0–10%                   1391  0.401249  0.193302  0.107  0.845         1391   
>10–25%                   763  0.433505  0.203580  0.142  0.873          763   
>25–50%                   967  0.431633  0.198858  0.123  0.855          967   
>50–75%                   897  0.419899  0.199188  0.091  0.898          897   
>75–90%                   735  0.412261  0.197698  0.089  0.826          735   
>90

In [33]:
print("\n=== CORRELACIÓN NUBOSIDAD VS ÍNDICES ===\n")

for indice in indices_pro:
    
    correlacion = pro_validos[
        ["porcentaje_nubosidad", f"{indice}_promedio"]
    ].corr().iloc[0, 1]
    
    print(
        f"{indice.upper():6s} → "
        f"correlación = {correlacion:.4f}"
    )


=== CORRELACIÓN NUBOSIDAD VS ÍNDICES ===

NDVI   → correlación = 0.1135
EVI    → correlación = 0.0998
LAI    → correlación = 0.1336
MSAVI  → correlación = 0.0609


In [34]:
# ============================================================
# AUDITORÍA DE CONSISTENCIA Y RANGOS — PRO
# ============================================================

indices_pro = ["ndvi", "evi", "lai", "msavi"]

print("=== AUDITORÍA DE CONSISTENCIA PRO ===\n")

for indice in indices_pro:

    promedio = pro[f"{indice}_promedio"]
    minimo = pro[f"{indice}_min"]
    maximo = pro[f"{indice}_max"]
    std = pro[f"{indice}_std"]

    # Comprobaciones
    min_mayor_promedio = (minimo > promedio).sum()
    promedio_mayor_max = (promedio > maximo).sum()
    std_negativa = (std < 0).sum()

    print(f"--- {indice.upper()} ---")
    print(f"min > promedio : {min_mayor_promedio}")
    print(f"promedio > max : {promedio_mayor_max}")
    print(f"std < 0        : {std_negativa}")

    print("\nDistribución del promedio:")
    print(
        promedio.describe(
            percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
    )

    print("\n")

=== AUDITORÍA DE CONSISTENCIA PRO ===

--- NDVI ---
min > promedio : 0
promedio > max : 0
std < 0        : 0

Distribución del promedio:
count    35747.000000
mean         0.353691
std          0.174964
min          0.007000
1%           0.164000
5%           0.187000
25%          0.227000
50%          0.275000
75%          0.451000
95%          0.724000
99%          0.800000
max          0.907000
Name: ndvi_promedio, dtype: float64


--- EVI ---
min > promedio : 0
promedio > max : 0
std < 0        : 0

Distribución del promedio:
count    35747.000000
mean         0.214637
std          0.123494
min          0.028000
1%           0.091000
5%           0.110000
25%          0.133000
50%          0.160000
75%          0.252000
95%          0.486000
99%          0.595000
max          0.877000
Name: evi_promedio, dtype: float64


--- LAI ---
min > promedio : 0
promedio > max : 0
std < 0        : 0

Distribución del promedio:
count    35747.000000
mean         1.215302
std          0.612512


In [35]:
# ============================================================
# VALORES EXTREMOS / FUERA DE RANGO
# ============================================================

print("=== POSIBLES VALORES A REVISAR ===\n")

reglas = {
    "ndvi": (0, 1),
    "evi": (0, 1),
    "msavi": (0, 1),
    "lai": (0, None)
}

for indice, (limite_inf, limite_sup) in reglas.items():

    col = f"{indice}_promedio"

    serie = pro[col].dropna()

    if limite_inf is not None:
        fuera_inf = (serie < limite_inf).sum()
    else:
        fuera_inf = 0

    if limite_sup is not None:
        fuera_sup = (serie > limite_sup).sum()
    else:
        fuera_sup = 0

    print(f"{indice.upper()}:")
    print(f"  menores que {limite_inf}: {fuera_inf}")

    if limite_sup is not None:
        print(f"  mayores que {limite_sup}: {fuera_sup}")

    print()

=== POSIBLES VALORES A REVISAR ===

NDVI:
  menores que 0: 0
  mayores que 1: 0

EVI:
  menores que 0: 0
  mayores que 1: 0

MSAVI:
  menores que 0: 0
  mayores que 1: 0

LAI:
  menores que 0: 1



In [36]:
# ============================================================
# LOCALIZAR EL ÚNICO LAI NEGATIVO
# ============================================================

fila_lai_negativo = pro[
    pro["lai_promedio"] < 0
][[
    "ID_POLIGONO",
    "fecha_captura_dt",
    "porcentaje_nubosidad",
    "ndvi_promedio",
    "evi_promedio",
    "lai_promedio",
    "lai_std",
    "lai_min",
    "lai_max",
    "msavi_promedio"
]]

print("=== ÚNICA OBSERVACIÓN CON LAI NEGATIVO ===\n")
print(fila_lai_negativo.to_string(index=False))

=== ÚNICA OBSERVACIÓN CON LAI NEGATIVO ===

ID_POLIGONO fecha_captura_dt  porcentaje_nubosidad  ndvi_promedio  evi_promedio  lai_promedio  lai_std  lai_min  lai_max  msavi_promedio
    AGC_148       2025-05-24                94.179          0.007         0.038        -0.112    0.034   -0.196    -0.04            0.01


In [37]:
print("\n=== COMPARACIÓN CON OBSERVACIONES DE LA MISMA PARCELA ===\n")

parcela = fila_lai_negativo["ID_POLIGONO"].iloc[0]

comparacion = (
    pro[pro["ID_POLIGONO"] == parcela]
    [[
        "ID_POLIGONO",
        "fecha_captura_dt",
        "porcentaje_nubosidad",
        "ndvi_promedio",
        "evi_promedio",
        "lai_promedio",
        "msavi_promedio"
    ]]
    .sort_values("fecha_captura_dt")
)

print(comparacion.to_string(index=False))


=== COMPARACIÓN CON OBSERVACIONES DE LA MISMA PARCELA ===

ID_POLIGONO fecha_captura_dt  porcentaje_nubosidad  ndvi_promedio  evi_promedio  lai_promedio  msavi_promedio
    AGC_148       2025-01-01                 0.000          0.276         0.144         0.966           0.147
    AGC_148       2025-01-03                 0.000          0.267         0.146         0.991           0.146
    AGC_148       2025-01-05                 0.000          0.174         0.130         0.616           0.108
    AGC_148       2025-01-06               100.000            NaN           NaN           NaN             NaN
    AGC_148       2025-01-07               100.000            NaN           NaN           NaN             NaN
    AGC_148       2025-01-08                 0.000          0.259         0.139         0.908           0.139
    AGC_148       2025-01-10               100.000            NaN           NaN           NaN             NaN
    AGC_148       2025-01-11               100.000          